# Supervised Learning. Decision Trees and ensembles

## Imports

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from category_encoders import CountEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from warnings import filterwarnings
filterwarnings('ignore')
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool

## Preprocessing    

In [118]:
df_raw = pd.read_csv('data/training.csv',index_col='RefId')
df_raw['PurchDate']= pd.to_datetime(df_raw['PurchDate'])



In [119]:

def train_val_test_date_split(
    df: pd.DataFrame,
    date_col: str,
    validation_date: str,
    test_date: str,
    drop_na_dates: bool = True,
    sort_by_time: bool = True,
):
    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in X")

    dates = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    cut_val  = pd.to_datetime(validation_date, utc=True)
    cut_test = pd.to_datetime(test_date,      utc=True)

    if not (cut_val < cut_test):
        raise ValueError("validation_date must be strictly earlier than test_date")


    if drop_na_dates:
        keep = dates.notna()
        df,  dates = df.loc[keep], dates.loc[keep]

    if sort_by_time:
        order = dates.sort_values().index
        df, dates = df.loc[order], dates.loc[order]


    m_train = dates <  cut_val
    m_val   = (dates >= cut_val) & (dates <  cut_test)
    m_test  = dates >= cut_test

    return df.loc[m_train], df.loc[m_val], df.loc[m_test]


In [120]:
val_cut = np.quantile(df_raw['PurchDate'],1/3)
test_cut = np.quantile(df_raw['PurchDate'],2/3)

In [121]:
df_train, df_val, df_test = train_val_test_date_split(df_raw,'PurchDate',val_cut,test_cut)

### Encoding categorical variables   

In [122]:
categorical_cols = df_train.select_dtypes(include=[object,'category']).columns
num_cols = df_train.drop(columns='IsOnlineSale').select_dtypes(include=[np.number]).columns


In [123]:
low = []
high = []
for col in categorical_cols:
    if len(df_train[col].unique())>20 :
        high.append(col)
    else:
        low.append(col)

    

In [124]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols.drop('IsBadBuy')),
        ("num_nan", MissingIndicator(features="missing-only", error_on_new=False), num_cols.drop('IsBadBuy')),
        ("ohe", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), low),
        ("ohe_nan", MissingIndicator(features="missing-only",error_on_new=False), low),
        ("cnt", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", CountEncoder()),
        ]), high),
        ("cnt_nan", MissingIndicator(features="missing-only",error_on_new=False), high),
    ],
    remainder='drop', 
    verbose_feature_names_out=True

)


In [125]:
encoded_train = preprocess.fit_transform(df_train)
encoded_val = preprocess.transform(df_val)
encoded_test = preprocess.transform(df_test)
columns = preprocess.get_feature_names_out()

In [126]:
X_train = pd.DataFrame(encoded_train,columns=columns)
X_val = pd.DataFrame(encoded_val,columns=columns)
X_test = pd.DataFrame(encoded_test,columns=columns)


In [127]:
y_train = df_train['IsBadBuy']
y_val = df_val['IsBadBuy']
y_test = df_test['IsBadBuy']


In [128]:
def gini_score(y_true,y_pred):
    return roc_auc_score(y_true,y_pred)*2-1

## DecisionTreeClassifier and DecisionTreeRegressor implementaition

In [264]:

class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None,
                 info_gain=None, value=None, proba=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.info_gain = info_gain
        self.value = value
        self.proba = proba

    def is_leaf(self):
        return self.value is not None
    
class BaseDecisionTree21():

    def __init__(self,
                impurity_fn,
                max_depth=np.inf,
                min_samples_split=2,
                min_samples_leaf=1,
                eps=1e-10,
                splitter="best",                        
                n_random_thresholds_per_feature=16,
                max_features=None,                      
                random_state=None,
                **kwargs):
        
        self.root = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.eps = float(eps)
        self.splitter = splitter
        self.max_features = max_features
        self.random_state = random_state
        self._rng = np.random.RandomState(random_state)
        self._impurity_fn = impurity_fn 

        self.classes_ = None
    def _resolve_min_samples(self, n_samples):

        if isinstance(self.min_samples_split, float):
            min_split = max(2, int(np.ceil(self.min_samples_split * n_samples)))
        else:
            min_split = int(self.min_samples_split)

        if isinstance(self.min_samples_leaf, float):
            min_leaf = max(1, int(np.ceil(self.min_samples_leaf * n_samples)))
        else:
            min_leaf = int(self.min_samples_leaf)

        return min_split, min_leaf

    def fit(self, X, y):
        self.X_ = np.asarray(X)
        self.y_ = np.asarray(y).ravel() 
        idx = np.arange(self.X_.shape[0])
        self.root = self._build_tree(idx, curr_depth=0)
        return self
    
    def predict(self, X):
        X = np.asarray(X)
        return np.asarray([self._make_prediction(x, self.root) for x in X])
    
    def _select_features(self, num_features):
        if self.max_features is None:
            k = num_features

        elif isinstance(self.max_features, int):
            k = max(1, min(self.max_features, num_features))

        elif isinstance(self.max_features, float):
            k = max(1, int(np.ceil(self.max_features * num_features)))

        else:
            raise ValueError("max_features должен быть None, int или float")

        idx = np.arange(num_features)
        if k == num_features:
            return idx
        return self._rng.choice(idx, size=k, replace=False)
    
    def _build_tree(self, idx, curr_depth=0):
        num_samples = idx.size
        min_split, _ = self._resolve_min_samples(num_samples)
        if (num_samples >= min_split) and (curr_depth < self.max_depth):
            best = self._get_best_split(idx)

            if (best is not None) and (best["info_gain"] > self.eps):
                left_subtree = self._build_tree(best["left_idx"], curr_depth + 1)
                right_subtree = self._build_tree(best["right_idx"], curr_depth + 1)
                return Node(feature_index=best["feature_index"],
                            threshold=float(best["threshold"]),
                            left=left_subtree,
                            right=right_subtree,
                            info_gain=float(best["info_gain"]),
                            value=None)

        leaf_value, proba = self._calculate_leaf_value(self.y_[idx])
        return Node(value=leaf_value, proba=proba)
    

    def _get_best_split(self, idx):
        best = None
        max_gain = -np.inf
        n_parent = idx.size
        y_parent = self.y_[idx]
        g_parent = self._impurity(y_parent)
        num_features = self.X_.shape[1] 
        features = self._select_features(num_features)
        _, min_leaf = self._resolve_min_samples(idx.size)
        for j in features:
            x = self.X_[idx, j]
            vals = np.unique(x)
            if vals.size < 2:
                continue
            thresholds = self._threshold(vals)

            for thr in thresholds:
                left_idx = idx[x<=thr]
                right_idx = idx[x>thr]

                if (left_idx.size < min_leaf) or (right_idx.size < min_leaf):
                    continue
                wL = left_idx.size/n_parent
                wR = right_idx.size/n_parent
                left_gini = self._impurity(self.y_[left_idx])
                right_gini = self._impurity(self.y_[right_idx])
                gain = g_parent - (wL*left_gini+wR*right_gini)
                if gain > max_gain:
                    max_gain = gain
                    best = {
                        "feature_index": j,
                        "threshold": float(thr),
                        "left_idx": left_idx,
                        "right_idx": right_idx,
                        "info_gain": float(gain),
                    }
        return best
    def _threshold(self, vals):
        if self.splitter == 'best':
            thresholds = (vals[:-1] + vals[1:]) * 0.5
        elif self.splitter == 'random':
            thresholds = np.array(self._rng.choice((vals[:-1] + vals[1:]) * 0.5))
        else: 
            raise ValueError('splitter должен быть "best" или "random"')
        return thresholds
    
    def _impurity(self, y):
        return self._impurity_fn(y)
    
    def _make_prediction(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._make_prediction(x, node.left)
        else:
            return self._make_prediction(x, node.right)



In [265]:

class DecisionTreeClassifier21(BaseDecisionTree21):
    def __init__(self, **kwargs):
        super().__init__(impurity_fn=self._gini, **kwargs)
        self.classes_ = None
    def fit(self, X, y):
        y = np.asarray(y).ravel()
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        return super().fit(X, y_idx.astype(np.int32, copy=False))
    def predict(self, X):
        idx_preds = super().predict(X)
        return self.classes_[idx_preds]

    def predict_proba(self, X):
        X = np.asarray(X)
        probas = [self._predict_proba_row(x, self.root) for x in X]
        return np.vstack(probas)

    @staticmethod
    def _gini(y):
        if len(y) == 0:
            return 0.0
        _, counts = np.unique(y, return_counts=True)
        p = counts / counts.sum()
        return 1.0 - np.sum(p ** 2)

    def _calculate_leaf_value(self,y_idx):
        counts = np.bincount(y_idx)
        probs = counts / counts.sum()
        leaf_val = int(np.argmax(counts))         
        proba = np.zeros(len(self.classes_), dtype=float)
        nz = np.nonzero(counts)[0]
        proba[nz] = probs[nz]
        return leaf_val, proba

    def _predict_proba_row(self, x, node):
        if node.is_leaf():
            return node.proba
        if x[node.feature_index] <= node.threshold:
            return self._predict_proba_row(x, node.left)
        else:
            return self._predict_proba_row(x, node.right)


In [266]:
class DecisionTreeRegressor21(BaseDecisionTree21):
    def __init__(self, **kwargs):
        super().__init__(impurity_fn=self._mse, **kwargs)
    @staticmethod
    def _mse(y):
        return np.var(y)
    def _calculate_leaf_value(self, y):
        return float(np.mean(y)),None
    
        

### Comparing with sklearn DecisionTreeClassifier

In [267]:
dtreeclf = DecisionTreeClassifier21(max_depth=7,min_samples_leaf = 0.009448090850413616, min_samples_split =0.006817138712941572,random_state=21)
dtreeclf.fit(X_train,y_train)
dtreeclf21_y_proba = dtreeclf.predict_proba(X_val)[:,1]
gini_score(y_val,dtreeclf21_y_proba)

0.45094228708539386

In [133]:
dtreeclf = DecisionTreeClassifier(max_depth=7,min_samples_leaf = 0.009448090850413616, min_samples_split =0.006817138712941572,random_state=21)
dtreeclf.fit(X_train,y_train)
dtreeclf_y_pred = dtreeclf.predict(X_val)
dtreeclf_y_proba = dtreeclf.predict_proba(X_val)[:,1]
gini_score(y_val,dtreeclf_y_proba)

0.4571855008383796

### Gini score almost the same

## RandomForestClassifier implementation

In [134]:

class RandomForestClassifier21:
    def __init__(self, n_estimators=100, max_depth=None, random_state=21,
                 max_features=None, min_samples_split=2, min_samples_leaf=1,splitter = 'random'):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf

        self.splitter = splitter
        self.models = []


    def fit(self, X, y):
        self._rng = np.random.RandomState(self.random_state)
        self.models = []
        X = np.asarray(X)
        y = np.asarray(y)
        for idx in self._get_bootstrap(self.n_estimators, X.shape[0]):
            model = DecisionTreeClassifier(
                max_depth=self.max_depth,
                random_state=self._rng.randint(0,1e+7),
                max_features=self.max_features,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                splitter=self.splitter,

            )
            model.fit(X[idx], y[idx])
            self.models.append(model)
        return self

    def predict(self, X):
        preds = np.stack([m.predict(X) for m in self.models], axis=1)


        final = np.array([self._majority_vote(row) for row in preds])
        return final  

    def predict_proba(self, X):
        probs = np.stack([m.predict_proba(X) for m in self.models], axis=0)
        return probs.mean(axis=0) 
    def _get_bootstrap(self, n_estimators, n_samples):
        
        return [self._rng.choice(n_samples, size=n_samples, replace=True)
                for _ in range(n_estimators)]
    @staticmethod
    def _majority_vote(row):
        vals, cnts = np.unique(row, return_counts=True)
        return vals[np.argmax(cnts)]


### Comparing with sklearn RandomForestClassifier


In [135]:
rfclf21 = RandomForestClassifier21(n_estimators=58,max_depth=16,random_state=21,max_features=0.8619666211233609,min_samples_split=0.07289343010398516,min_samples_leaf = 0.00032623871240269577)
rfclf21.fit(X_train,y_train)
rf_proba21 = rfclf21.predict_proba(X_val)[:,1]
gini_score(y_val,rf_proba21)

0.48099468136234

In [164]:
rfclf = RandomForestClassifier(n_estimators=58,max_depth=16,random_state=21,max_features=0.8619666211233609,min_samples_split=0.07289343010398516,min_samples_leaf = 0.00032623871240269577)
rfclf.fit(X_train,y_train)
rf_proba = rfclf.predict_proba(X_val)[:,1]
gini_score(y_val,rf_proba)

0.4803873159629144

### Gini score almost the same


## GradientBoostingClassifier implementation

In [210]:

class GDBTClassifier21:
    def __init__(self,
                 n_estimators=100,
                  max_depth=4, random_state=21,
                 max_features=None, min_samples_split=2, min_samples_leaf=1,splitter = 'best',learning_rate=0.1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.lr = learning_rate
        self.splitter = splitter
        self.trees_ = []
        self.init_bias_ = 0.0
        self._rng = np.random.RandomState(random_state)

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).ravel().astype(float)   
        p0 = np.clip(y.mean(), 1e-6, 1-1e-6)
        self.init_bias_ = np.log(p0/(1-p0))

        F = np.full(X.shape[0], self.init_bias_, dtype=float)
        self.trees_ = []
        for m in range(self.n_estimators):
            p = self._sigmoid(F)
            residual = y - p
            tree = DecisionTreeRegressor21(
                max_depth=self.max_depth,
                max_features=self.max_features,
                random_state=self._rng.randint(0, 2**31-1),
                min_samples_split = self.min_samples_split,
                min_samples_leaf = self.min_samples_leaf,
                splitter = self.splitter
                
            ).fit(X, residual)
            update = tree.predict(X)
            F += self.lr * update
            self.trees_.append(tree)
        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        F = np.full(X.shape[0], self.init_bias_, dtype=float)
        for t in self.trees_:
            F += self.lr * t.predict(X)
        p = self._sigmoid(F)
        return np.column_stack([1.0 - p, p])

    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= 0.5).astype(int)
    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))


In [201]:
def objective_gbt21(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',2,6)

    learning_rate = trial.suggest_float('learning_rate', 1e-3, 1e+2,log=True)

    clf = GDBTClassifier21(n_estimators=n_estimators, max_depth=max_depth,
                            learning_rate=learning_rate, random_state=21)


    clf.fit(X_train, y_train)
    scores = clf.predict_proba(X_val)[:,1]        
    
    return gini_score(y_val, scores)

study_gbt21 = optuna.create_study(direction='maximize',
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=8))
study_gbt21.optimize(objective_gbt21, n_trials=100, show_progress_bar=True)

[I 2025-10-04 11:51:52,157] A new study created in memory with name: no-name-96ffe8bd-c71b-41f5-ae47-a686a257dc16


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-04 11:51:55,783] Trial 0 finished with value: 0.45032367802424833 and parameters: {'n_estimators': 53, 'max_depth': 3, 'learning_rate': 0.0035214327266992612}. Best is trial 0 with value: 0.45032367802424833.
[I 2025-10-04 11:52:07,896] Trial 1 finished with value: 0.48068033350065353 and parameters: {'n_estimators': 180, 'max_depth': 3, 'learning_rate': 0.11360718707511257}. Best is trial 1 with value: 0.48068033350065353.
[I 2025-10-04 11:52:17,596] Trial 2 finished with value: 0.4585750572269749 and parameters: {'n_estimators': 144, 'max_depth': 3, 'learning_rate': 0.01576059495174097}. Best is trial 1 with value: 0.48068033350065353.
[I 2025-10-04 11:52:28,891] Trial 3 finished with value: 0.46270208924441425 and parameters: {'n_estimators': 87, 'max_depth': 5, 'learning_rate': 0.8166812125048919}. Best is trial 1 with value: 0.48068033350065353.
[I 2025-10-04 11:52:39,437] Trial 4 finished with value: 0.127602685188152 and parameters: {'n_estimators': 191, 'max_depth': 

In [202]:
study_gbt21.best_params

{'n_estimators': 173, 'max_depth': 4, 'learning_rate': 0.3099662523021278}

### Comparing with GradientBoostingClassifier from sklearn

In [205]:
def objective_gbt(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',2,6)

    learning_rate = trial.suggest_float('learning_rate', 1e-3, 1e+2,log=True)

    clf = GradientBoostingClassifier(n_estimators=n_estimators, max_depth=max_depth,
                            learning_rate=learning_rate, random_state=21)


    clf.fit(X_train, y_train)
    scores = clf.predict_proba(X_val)[:,1]        
    
    return gini_score(y_val, scores)

study_gbt = optuna.create_study(direction='maximize',
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=8))
study_gbt.optimize(objective_gbt, n_trials=100, show_progress_bar=True)

[I 2025-10-04 12:21:18,563] A new study created in memory with name: no-name-1def5490-e4be-4a3a-944c-3b264d923d8e


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-04 12:21:25,609] Trial 0 finished with value: 0.4774570367522373 and parameters: {'n_estimators': 83, 'max_depth': 4, 'learning_rate': 0.01643621522232122}. Best is trial 0 with value: 0.4774570367522373.
[I 2025-10-04 12:21:49,401] Trial 1 finished with value: 0.4716172449884124 and parameters: {'n_estimators': 191, 'max_depth': 6, 'learning_rate': 0.02387388444158522}. Best is trial 0 with value: 0.4774570367522373.
[I 2025-10-04 12:22:05,438] Trial 2 finished with value: 0.4740227655966498 and parameters: {'n_estimators': 130, 'max_depth': 6, 'learning_rate': 0.0016421027089499936}. Best is trial 0 with value: 0.4774570367522373.
[I 2025-10-04 12:22:24,511] Trial 3 finished with value: -0.24541972084865626 and parameters: {'n_estimators': 182, 'max_depth': 5, 'learning_rate': 27.92391299747844}. Best is trial 0 with value: 0.4774570367522373.
[I 2025-10-04 12:22:32,078] Trial 4 finished with value: 0.451732695737614 and parameters: {'n_estimators': 72, 'max_depth': 5, 'le

In [212]:
gdbtClf21 = GDBTClassifier21(**study_gbt21.best_params)
gdbtClf21.fit(X_train,y_train)
gdbt21_proba = gdbtClf21.predict_proba(X_val)[:,1]
gini_score(y_val,gdbt21_proba)



0.48748171050804734

In [213]:
gdbtClf = GradientBoostingClassifier(**study_gbt.best_params).fit(X_train,y_train)
gdbt_proba = gdbtClf.predict_proba(X_val)[:,1]
gini_score(y_val,gdbt_proba)

0.4862485567173016

### Gini score almost the same


### Comparing *LGBMClassifier*, *XGBClassifier* and *CatBoostClassifier*

In [152]:
X_train_raw, X_val_raw, X_test_raw = df_train.drop(columns=['IsBadBuy','PurchDate']), df_val.drop(columns=['IsBadBuy','PurchDate']), df_test.drop(columns=['IsBadBuy','PurchDate'])

In [153]:

cat_cols = list(X_train_raw.select_dtypes(include=['object','string','category']).columns)

X_train_fix = X_train_raw.copy()
X_val_fix   = X_val_raw.copy()
X_test_fix  = X_test_raw.copy()
for df_ in (X_train_fix, X_val_fix, X_test_fix):
    df_[cat_cols] = df_[cat_cols].astype('string').fillna('__NA__')
cat_cols


['Auction',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelType',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'PRIMEUNIT',
 'AUCGUART',
 'VNST']

In [154]:
def objective_cat(trial):
    params = dict(
        loss_function="Logloss", random_seed=21, verbose=False,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        iterations=trial.suggest_int("iterations", 600, 2000),
        depth=trial.suggest_int("depth", 4, 8),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
    )
    model = CatBoostClassifier(**params)
    model.fit(Pool(X_train_fix, y_train, cat_features=cat_cols),
              eval_set=Pool(X_val_fix, y_val, cat_features=cat_cols),
              use_best_model=True, early_stopping_rounds=100)
    p = model.predict_proba(X_val_fix)[:,1]
    return gini_score(y_val, p)

study_cat = optuna.create_study(direction='maximize',
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=8))
study_cat.optimize(objective_cat, n_trials=100, show_progress_bar=True)



[I 2025-10-02 19:25:46,326] A new study created in memory with name: no-name-ca083394-1223-4955-96cc-2ecc37999ff5


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-02 19:25:49,490] Trial 0 finished with value: 0.4920437001674387 and parameters: {'learning_rate': 0.1282199623969212, 'iterations': 1957, 'depth': 7, 'l2_leaf_reg': 4.405440863701996}. Best is trial 0 with value: 0.4920437001674387.
[I 2025-10-02 19:25:55,457] Trial 1 finished with value: 0.4908990271710949 and parameters: {'learning_rate': 0.03419398697535812, 'iterations': 1866, 'depth': 5, 'l2_leaf_reg': 1.4135899489810966}. Best is trial 0 with value: 0.4920437001674387.
[I 2025-10-02 19:25:59,688] Trial 2 finished with value: 0.47913748951283797 and parameters: {'learning_rate': 0.11288136480433095, 'iterations': 1229, 'depth': 8, 'l2_leaf_reg': 2.7462722903080135}. Best is trial 0 with value: 0.4920437001674387.
[I 2025-10-02 19:26:03,015] Trial 3 finished with value: 0.48112865597005783 and parameters: {'learning_rate': 0.15253888818673936, 'iterations': 704, 'depth': 8, 'l2_leaf_reg': 9.198291227825177}. Best is trial 0 with value: 0.4920437001674387.
[I 2025-10-02 

In [155]:
study_cat.best_params

{'learning_rate': 0.04077881010787122,
 'iterations': 838,
 'depth': 6,
 'l2_leaf_reg': 6.328015838174236}

In [298]:
def objective_light(trial):
    params = dict(
        objective="binary", metric="auc", random_state=21, n_jobs=-1,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        n_estimators=trial.suggest_int("n_estimators", 400, 1500),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 200),
        feature_fraction=trial.suggest_float("feature_fraction", 0.7, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.7, 1.0),
        bagging_freq=trial.suggest_int("bagging_freq", 1, 5),
    )
    model = LGBMClassifier(**params,verbosity=-1)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              eval_metric="auc",
              callbacks=[early_stopping(100, verbose=False)],
              )
    p = model.predict_proba(X_val)[:,1]
    return gini_score(y_val, p)
study_light = optuna.create_study(direction='maximize',
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=8))
study_light.optimize(objective_light, n_trials=100, show_progress_bar=True)

[I 2025-10-04 17:44:30,020] A new study created in memory with name: no-name-1ec8401d-0800-41c0-890d-257abe7aa9dd


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-04 17:44:31,540] Trial 0 finished with value: 0.46951436874701136 and parameters: {'learning_rate': 0.18866559332288915, 'n_estimators': 799, 'num_leaves': 90, 'min_child_samples': 153, 'feature_fraction': 0.7305860002941175, 'bagging_fraction': 0.7664658441269608, 'bagging_freq': 4}. Best is trial 0 with value: 0.46951436874701136.
[I 2025-10-04 17:44:33,836] Trial 1 finished with value: 0.4681640465078196 and parameters: {'learning_rate': 0.08626768482289683, 'n_estimators': 1470, 'num_leaves': 137, 'min_child_samples': 27, 'feature_fraction': 0.8958161503126674, 'bagging_fraction': 0.7782533323542628, 'bagging_freq': 3}. Best is trial 0 with value: 0.46951436874701136.
[I 2025-10-04 17:44:35,979] Trial 2 finished with value: 0.4596133752701148 and parameters: {'learning_rate': 0.12340728046311758, 'n_estimators': 1299, 'num_leaves': 215, 'min_child_samples': 103, 'feature_fraction': 0.7007195770912845, 'bagging_fraction': 0.7730059350437386, 'bagging_freq': 4}. Best is tr

In [301]:
study_light.best_params

{'learning_rate': 0.030967475606979888,
 'n_estimators': 437,
 'num_leaves': 39,
 'min_child_samples': 83,
 'feature_fraction': 0.8221716128552531,
 'bagging_fraction': 0.8146330125155706,
 'bagging_freq': 5}

In [158]:


def objective_xgb_dart(trial):
    params = dict(         
        random_state=21, n_jobs=-1,

        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        n_estimators=trial.suggest_int("n_estimators", 400, 1500),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 1e-1, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),

        sample_type=trial.suggest_categorical("sample_type", ["uniform", "weighted"]),
        normalize_type=trial.suggest_categorical("normalize_type", ["tree", "forest"]),
        rate_drop=trial.suggest_float("rate_drop", 0.0, 0.5),   
        skip_drop=trial.suggest_float("skip_drop", 0.0, 0.5),   
        one_drop=trial.suggest_categorical("one_drop", [0, 1])  
    )

    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
    )
    p = model.predict_proba(X_val)[:, 1]
    return gini_score(y_val, p)

study_xgb_dart = optuna.create_study(direction="maximize",
                                     pruner=optuna.pruners.MedianPruner(n_startup_trials=8))
study_xgb_dart.optimize(objective_xgb_dart, n_trials=300, show_progress_bar=True)


[I 2025-10-02 19:40:18,440] A new study created in memory with name: no-name-dd796a92-308a-488b-b57e-37b3dd50c280


  0%|          | 0/300 [00:00<?, ?it/s]

[I 2025-10-02 19:40:19,637] Trial 0 finished with value: 0.42444305307839003 and parameters: {'learning_rate': 0.1817482612218437, 'n_estimators': 416, 'max_depth': 9, 'min_child_weight': 7, 'subsample': 0.9393452108170428, 'colsample_bytree': 0.6694713813679063, 'reg_alpha': 0.00023284084544293362, 'reg_lambda': 0.03446323415670802, 'sample_type': 'uniform', 'normalize_type': 'forest', 'rate_drop': 0.0026123730989424665, 'skip_drop': 0.49654069171767434, 'one_drop': 1}. Best is trial 0 with value: 0.42444305307839003.
[I 2025-10-02 19:40:23,237] Trial 1 finished with value: 0.45811216843064595 and parameters: {'learning_rate': 0.0277271095902207, 'n_estimators': 1030, 'max_depth': 10, 'min_child_weight': 11, 'subsample': 0.6367445320233346, 'colsample_bytree': 0.9911053707736948, 'reg_alpha': 1.4708140782282206e-07, 'reg_lambda': 3.348110522786737, 'sample_type': 'uniform', 'normalize_type': 'forest', 'rate_drop': 0.48134986495285936, 'skip_drop': 0.1029707148975848, 'one_drop': 1}. B

In [300]:
study_xgb_dart.best_params

{'learning_rate': 0.02320585483555243,
 'n_estimators': 708,
 'max_depth': 4,
 'min_child_weight': 9,
 'subsample': 0.8187715482906502,
 'colsample_bytree': 0.9920291624414987,
 'reg_alpha': 1.38687089228864e-08,
 'reg_lambda': 0.006256952796128947,
 'sample_type': 'weighted',
 'normalize_type': 'tree',
 'rate_drop': 0.3572037195401061,
 'skip_drop': 0.2610549978645805,
 'one_drop': 0}

In [160]:
catboostclf = CatBoostClassifier(**study_cat.best_params)
catboostclf.fit(Pool(X_train_fix,y_train,cat_features=cat_cols),
              eval_set=Pool(X_val_fix, y_val, cat_features=cat_cols),
              use_best_model=True, early_stopping_rounds=100)
cat_y = catboostclf.predict_proba(X_val_fix)[:,1]


0:	learn: 0.6550797	test: 0.6552668	best: 0.6552668 (0)	total: 16.7ms	remaining: 14s
1:	learn: 0.6203919	test: 0.6185616	best: 0.6185616 (1)	total: 29.1ms	remaining: 12.2s
2:	learn: 0.5897973	test: 0.5890658	best: 0.5890658 (2)	total: 43.9ms	remaining: 12.2s
3:	learn: 0.5623416	test: 0.5627164	best: 0.5627164 (3)	total: 56.6ms	remaining: 11.8s
4:	learn: 0.5375934	test: 0.5388625	best: 0.5388625 (4)	total: 65.9ms	remaining: 11s
5:	learn: 0.5149170	test: 0.5171249	best: 0.5171249 (5)	total: 77.8ms	remaining: 10.8s
6:	learn: 0.4949271	test: 0.4980934	best: 0.4980934 (6)	total: 89.6ms	remaining: 10.6s
7:	learn: 0.4763755	test: 0.4796187	best: 0.4796187 (7)	total: 101ms	remaining: 10.5s
8:	learn: 0.4608565	test: 0.4656888	best: 0.4656888 (8)	total: 106ms	remaining: 9.74s
9:	learn: 0.4464367	test: 0.4527299	best: 0.4527299 (9)	total: 118ms	remaining: 9.76s
10:	learn: 0.4333653	test: 0.4408044	best: 0.4408044 (10)	total: 133ms	remaining: 10s
11:	learn: 0.4203474	test: 0.4286654	best: 0.428665

In [302]:
LGBMclf = LGBMClassifier(**study_light.best_params)
LGBMclf.fit(X_train,y_train,eval_set=(X_val,y_val), eval_metric='auc',
            callbacks=[early_stopping(100, verbose=False)])
light_y = LGBMclf.predict_proba(X_val)[:,1]


In [240]:
XGBclf = XGBClassifier(**study_xgb_dart.best_params)
XGBclf.fit(X_train,y_train)
xgb_y = XGBclf.predict_proba(X_val)[:,1]


In [304]:
print(gini_score(y_val,cat_y))
print(gini_score(y_val,light_y))
print(gini_score(y_val,xgb_y))

0.493927740338461
0.48940538532829736
0.49364731629518066


Review the documentation of the libraries and fine-tune the algorithms for the task.

Note key differences between each implementation.

Analyze special features of each algorithm (how does "categorical feature" work in Catboost, what is DART mode in XGBoost)?

Which GBDT model gives the best result? 

## Отличия реализаций

- **CatBoost**
  - Нативная работа с категориальными признаками (target encoding с перестановками).
  - Использует симметричные деревья.
  - Ordered boosting предотвращает утечку таргета и переобучение.
  - Удобен, когда много категориальных фич.

- **LightGBM**
  - Очень быстрая реализация GBDT.
  - Основан на histogram-based splits (ускоряет обучение).
  - Использует leaf-wise рост деревьев (может переобучаться на малых данных).
  - Хорош для больших датасетов.

- **XGBoost**
  - Классическая реализация бустинга.
  - По умолчанию level-wise рост деревьев.
  - **DART mode**: dropout деревьев во время обучения (аналог dropout в нейросетях), снижает переобучение на шумных данных.
- **В данном случае CatBoost и XGBoost оказались лучшими моделями по качеству (Gini ≈ 0.494).**



In [310]:
print(f'Gini score on train dataset - {gini_score(y_train, catboostclf.predict_proba(X_train_fix)[:,1])}')
print(f'Gini score on validation dataset - {gini_score(y_val, catboostclf.predict_proba(X_val_fix)[:,1])}')
print(f'Gini score on test dataset - {gini_score(y_test, catboostclf.predict_proba(X_test_fix)[:,1])}')

Gini score on train dataset - 0.6215307007989788
Gini score on validation dataset - 0.493927740338461
Gini score on test dataset - 0.46922802238106653


# Оценка лучшей модели на train / validation / test

## Результаты CatBoost (лучшая модель)

- **Train Gini**: 0.6215  
- **Validation Gini**: 0.4939  
- **Test Gini**: 0.4692  

## Анализ
- Метрика на **train** заметно выше (0.62), чем на валидации (0.49) и тесте (0.47). Это ожидаемо: модель подстраивается под обучающие данные.
- Качество на **валидации и тесте близкое** (разница ~0.025).  
- Это значит, что **падение качества от валидации к тесту небольшое**, модель сохраняет способность к обобщению.

## Вывод: cильного переобучения нет


### Implement the ExtraTreesClassifier and check its performance. You must improve the result of a single tree and obtain a Gini score of at least 0.12 on the validation dataset.

In [272]:
class ExtraTreesClassifier21:
    def __init__(self,
                 n_estimators=100,
                 max_depth=None,
                 max_features=None,
                 min_samples_split=2,
                 min_samples_leaf=1,
                 n_random_thresholds_per_feature=16,
                 bootstrap=False,
                 random_state=21):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.n_random_thresholds_per_feature = n_random_thresholds_per_feature
        self.bootstrap = bootstrap
        self.random_state = random_state
        self.models = []

    def fit(self, X, y):
        self._rng = np.random.RandomState(self.random_state)
        X = np.asarray(X); y = np.asarray(y)
        n = X.shape[0]
        self.models = []

        for _ in range(self.n_estimators):
            if self.bootstrap:
                idx = self._rng.choice(n, size=n, replace=True)
            else:
                idx = np.arange(n)  # ExtraTrees — без бутстрапа по умолчанию

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                max_features=self.max_features,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                splitter="random",
                random_state=self._rng.randint(0, 2**31-1),
            )
            tree.fit(X[idx], y[idx])
            self.models.append(tree)
        return self

    def predict_proba(self, X):
        probs = np.stack([m.predict_proba(X) for m in self.models], axis=0)
        return probs.mean(axis=0)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [295]:
et21 = ExtraTreesClassifier21(n_estimators=275,max_features=0.66, random_state=21,max_depth=9).fit(X_train, y_train)
print(gini_score(y_train, et21.predict_proba(X_train)[:,1]))
print(gini_score(y_val, et21.predict_proba(X_val)[:,1]))
print(gini_score(y_test, et21.predict_proba(X_test)[:,1]))




0.6506991900683379
0.4859654033418892
0.4204828903450739


In [297]:
et = ExtraTreesClassifier(n_estimators=275,max_features=0.66, random_state=21,max_depth=9).fit(X_train, y_train)
print(gini_score(y_train, et.predict_proba(X_train)[:,1]))
print(gini_score(y_val, et.predict_proba(X_val)[:,1]))
print(gini_score(y_test, et.predict_proba(X_test)[:,1]))


0.6506991900683379
0.4859654033418892
0.4204828903450739


## ExtraTrees (Extremely Randomized Trees)

### Это ансамбль деревьев решений, похожий на RandomForest, но с большей случайностью.

### Как работает
- Для каждого узла выбирается случайное подмножество признаков.  
- Для каждого признака случайным образом выбирается порог (а не ищется оптимальный).  
- Среди этих случайных кандидатов выбирается лучший по impurity.  
- Строится ансамбль из многих таких деревьев, предсказание усредняется.

### Плюсы
- **Быстрее обучение**, чем у RandomForest (нет перебора всех порогов).  
- **Меньше переобучение** за счёт высокой случайности.  
- Хорошо работает на **шумных данных**.  
- **Простая настройка**: меньше гиперпараметров, устойчивость к выбору параметров.  

### Минусы
- **Может проигрывать в точности** RandomForest на чистых и структурированных данных.  
- Из-за сильной случайности иногда нужно **большее число деревьев** для хорошей стабильности.  
- Интерпретация немного хуже (деревья менее осмысленные).  